In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [136]:
# loading datasets
fews_dt =  pd.read_excel("data/raw/FEWS_NET_Staple_Food_Price_Data.xlsx")
wfp_dt = pd.read_csv("data/raw/wfp_food_prices_nga.csv")

In [137]:
fews_dt.head()
# loading the top data from fews datasets

,country,market,admin_1,longitude,latitude,cpcv2,product,source_document,period_date,price_type,product_source,unit,unit_type,currency,value
0,Nigeria,Aba,Abia,7.37063,5.10719,P23490AA,Bread,Famine Early Warning Systems Network (FEWS NET...,2008-07-14,Retail,Local,ea,Item,NGN,NaN
1,Nigeria,Aba,Abia,7.37063,5.10719,P23490AA,Bread,Famine Early Warning Systems Network (FEWS NET...,2008-08-15,Retail,Local,ea,Item,NGN,NaN
2,Nigeria,Aba,Abia,7.37063,5.10719,P23490AA,Bread,Famine Early Warning Systems Network (FEWS NET...,2008-09-15,Retail,Local,ea,Item,NGN,NaN
3,Nigeria,Aba,Abia,7.37063,5.10719,P23490AA,Bread,Famine Early Warning Systems Network (FEWS NET...,2008-12-16,Retail,Local,ea,Item,NGN,NaN
4,Nigeria,Aba,Abia,7.37063,5.10719,P23490AA,Bread,Famine Early Warning Systems Network (FEWS NET...,2009-01-23,Retail,Local,ea,Item,NGN,NaN


In [138]:
wfp_dt.head() # loading the top data in the wfp datasets

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,1/15/2002,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Maize,51,KG,actual,Wholesale,NGN,175.92,1.54
1,1/15/2002,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Millet,73,KG,actual,Wholesale,NGN,150.18,1.31
2,1/15/2002,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Rice (imported),64,KG,actual,Wholesale,NGN,358.70,3.14
3,1/15/2002,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Sorghum,65,KG,actual,Wholesale,NGN,155.61,1.36
4,1/15/2002,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,pulses and nuts,Beans (niebe),120,KG,actual,Wholesale,NGN,196.87,1.72


In [139]:
fews_dt['period_date'] =  pd.to_datetime(fews_dt['period_date'])  # converting period_date in fews_dt from obeject to datetime

In [140]:
wfp_dt['date'] =  pd.to_datetime(wfp_dt['date']) # converting date in wfp_dt from object to datetime  


In [141]:
fews_dt['date']= fews_dt['period_date'] # standardize the column in few_dt to match the column name in wfp_dt 'date'

In [142]:
fews_dt = fews_dt.drop('period_date',axis = 1) # remove the "period_date" since i have created another column 'date'

In [143]:
fews_dt['commodity'] = fews_dt['product'] 
# standardize the column in few_dt to match the column name in wfp_dt 'product'

In [144]:
fews_dt['price'] = fews_dt['value'] # standardize the column in few_dt to match the column name in wfp_dt 'price'

In [145]:
fews_dt = fews_dt.drop(columns=['product', 'value'])
# drop both column in few_dt..

In [146]:
null_by_commodity = fews_dt.groupby('commondity')['price'].apply(
    lambda x: (x.isnull().sum() / len(x)) * 100
).round(2)

print(null_by_commodity.sort_values(ascending=False)) #grouping the missing value which exist in price(few_dt) based on the product

commondity
Gari (Yellow)           28.06
Diesel                  26.72
Gasoline                26.60
Bread                   23.23
Rice (5% Broken)        22.60
Yams                    22.38
Cowpeas (Brown)         19.97
Gari (White)            19.31
Maize Grain (Yellow)    17.85
Rice (Milled)           16.87
Palm Oil (Refined)      16.45
Sorghum (White)         15.57
Sorghum (Brown)         15.34
Cowpeas (White)         14.54
Maize Grain (White)     14.54
Millet (Pearl)          14.47
Cattle (Male)            4.92
Groundnuts (Shelled)     4.31
Goats (Male)             2.61
Sheep (Male)             2.61
Name: price, dtype: float64


In [147]:
# Flag rows where price is null (1 = missing, 0 = present)
# I kept these rows for market/date coverage analysis
# but exclude them from price comparisons
fews_dt['price_missing'] = fews_dt['price'].isnull().astype(int)

In [148]:
fews_dt.info() # validating to see if it worked. and it did

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315712 entries, 0 to 315711
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   country          315712 non-null  object        
 1   market           315712 non-null  object        
 2   admin_1          315712 non-null  object        
 3   longitude        315712 non-null  float64       
 4   latitude         315712 non-null  float64       
 5   cpcv2            315712 non-null  object        
 6   source_document  315712 non-null  object        
 7   price_type       315712 non-null  object        
 8   product_source   315712 non-null  object        
 9   unit             315712 non-null  object        
 10  unit_type        315712 non-null  object        
 11  currency         315712 non-null  object        
 12  date             315712 non-null  datetime64[ns]
 13  commondity       315712 non-null  object        
 14  price            258

In [149]:
#  Replace underscores with spaces
fews_dt['unit'] = fews_dt['unit'].str.replace('_', ' ')

In [150]:
fews_dt['unit'].unique()

array(['ea', 'kg', '100 kg', 'l', '30 l', '50 kg', '100 tubers',
       '60 tubers'], dtype=object)

In [151]:
wfp_dt['unit'] = wfp_dt['unit'].str.lower() #standardise wfp unit from uppercase to lower. so it can match with fews_dt unit

In [152]:
wfp_dt['unit'].unique()

array(['kg', '100 kg', '50 kg', 'unit', '2.6 kg', '2.8 kg', '2.7 kg', 'l',
       '2.5 kg', '100 l', '2.1 kg', '400 g', '2.2 kg', '0.5 kg', '30 pcs',
       '1.3 kg', '300 g', '250 g', '500 g', '100 tubers'], dtype=object)

In [153]:
# Standardise FEWS units to lowercase and replace spaces with empty
fews_dt['commodity'] = fews_dt['commodity'].str.lower()

In [154]:
# Standardise FEWS units to lowercase and replace spaces with empty. so they can match
wfp_dt['commodity'] = wfp_dt['commodity'].str.lower()

In [155]:
fews_dt['commondity'].unique()

array(['bread', 'cowpeas (brown)', 'cowpeas (white)', 'diesel',
       'gari (white)', 'gari (yellow)', 'gasoline',
       'groundnuts (shelled)', 'maize grain (white)',
       'maize grain (yellow)', 'millet (pearl)', 'palm oil (refined)',
       'rice (5% broken)', 'rice (milled)', 'sorghum (brown)',
       'sorghum (white)', 'yams', 'cattle (male)', 'goats (male)',
       'sheep (male)'], dtype=object)

In [156]:
wfp_dt['commodity'].unique()

array(['maize', 'millet', 'rice (imported)', 'sorghum', 'beans (niebe)',
       'wheat', 'maize (white)', 'sorghum (white)',
       'rice (milled, local)', 'bread', 'cassava meal (gari, yellow)',
       'gari (white)', 'maize (yellow)', 'rice (local)',
       'sorghum (brown)', 'yam (abuja)', 'fuel (diesel)',
       'fuel (petrol-gasoline)', 'oil (palm)', 'cowpeas (brown)',
       'cowpeas (white)', 'yam', 'groundnuts (shelled)', 'maize flour',
       'meat (beef)', 'meat (goat)', 'milk (powder)', 'oil (vegetable)',
       'beans (red)', 'beans (white)', 'groundnuts', 'onions', 'fish',
       'eggs', 'bananas', 'oranges', 'spinach', 'watermelons', 'cowpeas',
       'tomatoes', 'salt', 'sugar', 'seasoning (maggi cube)'],
      dtype=object)

In [157]:
# Standardise FEWS units to lowercase and replace spaces 
fews_dt['market'] = fews_dt['market'].str.lower().str.replace(' ', '')

In [158]:
# Standardise FEWS units to lowercase and replace underscores with spaces
wfp_dt['market'] = wfp_dt['market'].str.lower().str.replace(' ', '')

In [159]:
wfp_dt['price_type' ] = wfp_dt['pricetype'] # change the column from (pricetype) to price_type. so it can match with wfp_dt

In [160]:
wfp_dt = wfp_dt.drop('pricetype',axis=1) # drop the column 'pricetype' . since we created another column

In [161]:
# Extract city name before the comma for FEWS markets
# to match WFP's simpler market naming convention
fews_dt['market'] = fews_dt['market'].str.split(',').str[0].str.strip()

In [162]:
fews_dt['market'].unique() # checking if it worked and it did

array(['aba', 'biu', 'damaturu', 'dandume', 'geidam', 'giwa', 'gombe',
       'gujungu', 'gwandu', 'ibadan', 'ilela', 'kano', 'kauranamoda',
       'lagos', 'maiadua', 'maiduguri', 'maigatari', 'mubi',
       'port-harcourt', 'potiskum', 'saminaka'], dtype=object)

In [163]:
wfp_dt['market'].unique() # checking if it worked

array(['jibia(cbm)', 'illela(cbm)', 'maiadoua(cbm)', 'damassack(cbm)',
       'dawanau', 'maigatari(cbm)', 'ibadan', 'maiduguri', 'lagos',
       'giwa', 'kauranamoda', 'aba', 'gombe', 'gujungu', 'saminaka',
       'dandume', 'gwandu', 'mubi', 'biu', 'damaturu', 'potiskum',
       'abbagamaram', 'bagaroad', 'bullunkutu', 'budum', 'custom',
       'kusawamshanu', 'monday', 'tashanbama', 'boloristores',
       'damaturu(sundaymarket)', 'geidam', 'gujba(buniyadi)', 'jakusko',
       'bade(gashua)', 'nguru', 'yunusari', 'yusufari', 'bursari',
       'gulani(tettaba)', 'gamboru', 'kashuwanshanu', 'bayantasha',
       'damagum', 'nangere', 'madagali', 'michika', 'damboa', 'monguno',
       'gwozacentral', 'pulka', 'bama', 'konduga', 'dikwacentral', 'mafa',
       'banki', 'machinacentral', 'shanimainmarket', 'rann',
       'magumericentral', 'ngala', 'gajiram', 'biriri', 'gombi',
       'mubinorth', 'ngalang', 'ngelzarma', 'jimetagrain'], dtype=object)

In [166]:
# Save cleaned datasets
fews_dt.to_csv('data/output/fews_cleaned.csv', index=False)
wfp_dt.to_csv('data/output/wfp_cleaned.csv',index=False)